In [28]:
import pandas as pd
import numpy as np
import time

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)

import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import phik

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import roc_auc_score

import joblib

In [29]:
classification_data = pd.read_csv(r'Data/financial_transactions.csv')
print(classification_data.info())
print(classification_data.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   amount           100000 non-null  float64
 1   weekday          100000 non-null  int64  
 2   month            100000 non-null  int64  
 3   description_len  100000 non-null  int64  
 4   is_income        100000 non-null  int64  
 5   category         100000 non-null  object 
dtypes: float64(1), int64(4), object(1)
memory usage: 4.6+ MB
None
     amount  weekday  month  description_len  is_income     category
0   4342.71        5      1               49          0  Развлечения
1  27619.44        5      7               33          1     Зарплата
2  14499.34        3      7               40          1     Зарплата
3   1302.40        4      4               15          0    Транспорт
4   4925.35        4      6               36          0    Транспорт


In [30]:
RANDOM_STATE = 42
y = classification_data['category']
X = classification_data.drop('category', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=RANDOM_STATE, test_size=0.1)

In [31]:
cat_cols = []
num_cols = [col for col in classification_data.select_dtypes(include=['int64', 'float64']).columns if col != 'category']

In [32]:
label_pipe = Pipeline([
    ('Impute', SimpleImputer(missing_values=np.nan, strategy='most_frequent')),
    ('label', LabelEncoder())
])

In [33]:
data_preprosessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols)
])

In [36]:
lr_pipe = Pipeline([
    ('preprosessor', data_preprosessor),
    ('model', LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE))
])

lr_auc = cross_val_score(lr_pipe, X, y, cv=5, scoring='roc_auc_ovo').mean()

l_start = time.time()
lr_pipe.fit(X_train, y_train)
l_finish = time.time()

p_start = time.time()
y_pred = lr_pipe.predict(X_test)
p_finish = time.time()

lr_stats = {
    'Время обучения': l_finish - l_start,
    'Время предсказания': p_finish - p_start,
    'ROC_AUC': lr_auc
}

for key, value in lr_stats.items():
    print(f'{key}: {value}')

Время обучения: 0.8822555541992188
Время предсказания: 0.006205081939697266
ROC_AUC: 0.7683334328188189
